In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

hu = pd.read_csv("data/processed/humaid_safety.csv")
t26 = pd.read_csv("data/processed/t26_safety.csv")
t26 = t26[t26["event"] != "2013_Sardinia_floods"]          # Italian

HU_HOLD = ["maryland_floods_2018", "srilanka_floods_2017"]
T26_TRAIN = ["2013_Alberta_floods", "2013_Colorado_floods"]

hu_train = hu[(~hu["event"].isin(HU_HOLD)) & (hu["split"] == "train")][["text", "safety_label"]]
t26_train = t26[t26["event"].isin(T26_TRAIN)][["text", "safety_label"]]

tests = {
    "HumAID unseen floods":   hu[hu["event"].isin(HU_HOLD)],
    "T26 Queensland":         t26[t26["event"] == "2013_Queensland_floods"],
    "T26 Philippines+Manila": t26[t26["event"].isin(["2012_Philipinnes_floods", "2013_Manila_floods"])],
}

def run(train, name, th=0.35):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(
        vec.fit_transform(train["text"]), train["safety_label"])
    hz = list(clf.classes_).index("hazard")
    rows = []
    for tn, d in tests.items():
        p = clf.predict_proba(vec.transform(d["text"]))[:, hz]
        pred = np.where(p >= th, "hazard", "non_hazard")
        pr, rc, f1, _ = precision_recall_fscore_support(
            d["safety_label"], pred, labels=["hazard"], zero_division=0)
        rows.append({"model": name, "test": tn,
                     "hazard_n": int((d["safety_label"] == "hazard").sum()),
                     "precision": round(pr[0], 2), "recall": round(rc[0], 2), "f1": round(f1[0], 2)})
    return rows

res = run(hu_train, "HumAID only") + run(pd.concat([hu_train, t26_train]), "HumAID + T26 (Alberta, Colorado)")
pd.DataFrame(res)

,model,test,hazard_n,precision,recall,f1
0,HumAID only,HumAID unseen floods,96,0.46,0.81,0.58
1,HumAID only,T26 Queensland,119,0.44,0.54,0.49
2,HumAID only,T26 Philippines+Manila,86,0.27,0.14,0.18
3,"HumAID + T26 (Alberta, Colorado)",HumAID unseen floods,96,0.42,0.91,0.58
4,"HumAID + T26 (Alberta, Colorado)",T26 Queensland,119,0.42,0.72,0.53
5,"HumAID + T26 (Alberta, Colorado)",T26 Philippines+Manila,86,0.31,0.28,0.29


In [2]:
from sklearn.metrics import average_precision_score

def ap_scores(train, name):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(
        vec.fit_transform(train["text"]), train["safety_label"])
    hz = list(clf.classes_).index("hazard")
    rows = []
    for tn, d in tests.items():
        y = (d["safety_label"] == "hazard").astype(int)
        p = clf.predict_proba(vec.transform(d["text"]))[:, hz]
        rows.append({"model": name, "test": tn,
                     "hazard_share": round(y.mean(), 3),
                     "avg_precision": round(average_precision_score(y, p), 3)})
    return rows

pd.DataFrame(ap_scores(hu_train, "HumAID only") +
             ap_scores(pd.concat([hu_train, t26_train]), "HumAID + T26"))

,model,test,hazard_share,avg_precision
0,HumAID only,HumAID unseen floods,0.104,0.732
1,HumAID only,T26 Queensland,0.154,0.475
2,HumAID only,T26 Philippines+Manila,0.080,0.214
3,HumAID + T26,HumAID unseen floods,0.104,0.737
4,HumAID + T26,T26 Queensland,0.154,0.551
5,HumAID + T26,T26 Philippines+Manila,0.080,0.263


## Findings

- Training on HumAID only vs HumAID + T26 (Alberta, Colorado floods), threshold fixed at 0.35 (chosen on HumAID dev).
- Hazard recall rose on all three unseen test sets (HumAID unseen floods 0.81 to 0.91, T26 Queensland 0.54 to 0.72, T26 Philippines/Manila 0.14 to 0.28); precision fell slightly on the first two (0.46 to 0.42, 0.44 to 0.42) and rose on the third (0.27 to 0.31).
- F1 gains are small (unchanged on HumAID unseen floods, +0.04 Queensland) except Philippines/Manila (0.18 to 0.29, only 86 hazard tweets).
- Part of the recall gain may come from the higher hazard share in training moving the model along the same precision-recall curve; average precision (below) checks this.
- Philippines/Manila remains the weakest case.
- Single run, small test sets: differences of a few points are not conclusive.
- - Average precision (threshold-independent): HumAID unseen floods 0.732 → 0.737 (flat), T26 Queensland 0.475 → 0.551, T26 Philippines/Manila 0.214 → 0.263 (no-skill levels: 0.104, 0.154, 0.080).
- Conclusion: mixing T26 into training improves ranking on T26 events; on HumAID unseen floods the higher recall was threshold movement only. Part of the gain may reflect T26's labelling style. Single run, small test sets.
-  Threshold re-chosen for the mixed model on a held-out dev set (HumAID dev + 20% of T26 Alberta/Colorado), not on the test sets. F2 peaks at 0.87 for thresholds 0.35 and 0.40; 0.40 chosen (precision 0.70, recall 0.93 on dev) as it ties on F2 with higher precision.
- Dev recall is optimistic because dev comes from the training events/datasets; see the unseen test results for the realistic numbers.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score

t_tr, t_dev = train_test_split(t26_train, test_size=0.2, random_state=0,
                               stratify=t26_train["safety_label"])
hu_dev = hu[(~hu["event"].isin(HU_HOLD)) & (hu["split"] == "dev")][["text", "safety_label"]]

train = pd.concat([hu_train, t_tr])
dev = pd.concat([hu_dev, t_dev])

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(
    vec.fit_transform(train["text"]), train["safety_label"])
hz = list(clf.classes_).index("hazard")
p_dev = clf.predict_proba(vec.transform(dev["text"]))[:, hz]

rows = []
for th in np.arange(0.20, 0.71, 0.05):
    pred = np.where(p_dev >= th, "hazard", "non_hazard")
    pr, rc, f1, _ = precision_recall_fscore_support(
        dev["safety_label"], pred, labels=["hazard"], zero_division=0)
    f2 = fbeta_score(dev["safety_label"], pred, beta=2, pos_label="hazard", zero_division=0)
    rows.append({"threshold": round(th, 2), "precision": round(pr[0], 2),
                 "recall": round(rc[0], 2), "f1": round(f1[0], 2), "f2": round(f2, 2)})
pd.DataFrame(rows)

,threshold,precision,recall,f1,f2
0,0.20,0.48,0.99,0.65,0.82
1,0.25,0.55,0.98,0.70,0.85
2,0.30,0.59,0.96,0.74,0.86
3,0.35,0.65,0.95,0.77,0.87
4,0.40,0.70,0.93,0.80,0.87
5,0.45,0.74,0.90,0.81,0.86
6,0.50,0.77,0.88,0.82,0.86
7,0.55,0.80,0.86,0.83,0.85
8,0.60,0.82,0.83,0.82,0.82
9,0.65,0.84,0.80,0.82,0.80


In [4]:
THRESHOLD = 0.40
rows = []
for tn, d in tests.items():
    y = (d["safety_label"] == "hazard").astype(int)
    p = clf.predict_proba(vec.transform(d["text"]))[:, hz]
    pred = np.where(p >= THRESHOLD, "hazard", "non_hazard")
    pr, rc, f1, _ = precision_recall_fscore_support(
        d["safety_label"], pred, labels=["hazard"], zero_division=0)
    rows.append({"test": tn, "hazard_n": int(y.sum()),
                 "precision": round(pr[0], 2), "recall": round(rc[0], 2),
                 "f1": round(f1[0], 2),
                 "avg_precision": round(average_precision_score(y, p), 3)})
pd.DataFrame(rows)

,test,hazard_n,precision,recall,f1,avg_precision
0,HumAID unseen floods,96,0.49,0.80,0.61,0.737
1,T26 Queensland,119,0.47,0.55,0.51,0.540
2,T26 Philippines+Manila,86,0.38,0.17,0.24,0.253


In [5]:
import os
import joblib

os.makedirs("data/models", exist_ok=True)
joblib.dump({"vec": vec, "clf": clf, "threshold": 0.40}, "data/models/safety_baseline_v2.joblib")
print("saved")

saved


## Baseline v2 — final results (threshold 0.40, chosen on dev, not re-tuned)

| Test | Hazard n | Precision | Recall | F1 | Avg precision |
                       |---   | ---| ---| ---| ---| -----|
| HumAID unseen floods | 96 | 0.49 | 0.80 | 0.61 | 0.737 |
| T26 Queensland | 119 | 0.47 | 0.55 | 0.51 | 0.540 |
| T26 Philippines+Manila | 86 | 0.38 | 0.17 | 0.24 | 0.253 |

- Dev recall (0.93) is optimistic: dev shares datasets/events with training. On unseen datasets recall ranges 0.17–0.80.
- Naive baseline ("always non_hazard") has zero hazard recall on every test.
- Limits: noisy labels, different annotation guidelines across datasets, 2012–2019 language, not NYC-specific, bag-of-words.
- Next model: fine-tuned language model on in-domain text.